In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# ─────────────────────────────────────────────
# 1. GENERATE DATASET WITH PANDAS
#    Simulates: Temperature (°C) vs Ice Cream Sales (units)
#    Relationship: Sales peak at ~30°C (non-linear)
# ─────────────────────────────────────────────
np.random.seed(42)

temperature = np.linspace(5, 45, 200)
sales = -2 * (temperature - 30)**2 + 500 + np.random.normal(0, 25, 200)

# Store in a DataFrame
df = pd.DataFrame({"temperature": temperature, "sales": sales})

print("=" * 50)
print("📋 DATASET OVERVIEW")
print("=" * 50)
print(df.head(10).to_string(index=False))
print(f"\nShape     : {df.shape}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nStatistics:\n{df.describe().round(2)}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print("=" * 50)

X = df[["temperature"]].values
y = df["sales"].values

# ─────────────────────────────────────────────
# 2. SPLIT INTO TRAIN / TEST USING PANDAS
# ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

df_train = pd.DataFrame({"temperature": X_train.flatten(), "sales": y_train})
df_test  = pd.DataFrame({"temperature": X_test.flatten(),  "sales": y_test})

print(f"\n✅ Train samples : {len(df_train)}")
print(f"✅ Test  samples : {len(df_test)}")

# ─────────────────────────────────────────────
# 3. TRAIN POLYNOMIAL REGRESSION MODELS
#    Degrees: 1 (linear), 2 (quadratic), 5 (higher-order)
# ─────────────────────────────────────────────
degrees = [1, 2, 5]
models  = {}

for deg in degrees:
    model = Pipeline([
        ("poly_features", PolynomialFeatures(degree=deg, include_bias=False)),
        ("linear_reg",    LinearRegression())
    ])
    model.fit(X_train, y_train)
    models[deg] = model

# ─────────────────────────────────────────────
# 4. EVALUATE MODELS — STORE RESULTS IN PANDAS
# ─────────────────────────────────────────────
results = []
for deg, model in models.items():
    y_pred = model.predict(X_test)
    results.append({
        "Degree"  : deg,
        "R² Score": round(r2_score(y_test, y_pred), 4),
        "RMSE"    : round(np.sqrt(mean_squared_error(y_test, y_pred)), 4)
    })

df_results = pd.DataFrame(results)
print("\n📊 MODEL EVALUATION RESULTS")
print("=" * 40)
print(df_results.to_string(index=False))
print("=" * 40)
print(f"\n🏆 Best Model: Degree {df_results.loc[df_results['R² Score'].idxmax(), 'Degree']}")

# ─────────────────────────────────────────────
# 5. VISUALIZATIONS (4 plots)
# ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Polynomial Regression — Temperature vs Ice Cream Sales",
             fontsize=15, fontweight="bold", y=1.01)

X_plot = np.linspace(5, 45, 300).reshape(-1, 1)   # smooth curve for plotting

colors = {1: "#e74c3c", 2: "#2ecc71", 5: "#9b59b6"}
labels = {1: "Degree 1 (Linear)", 2: "Degree 2 (Quadratic)", 5: "Degree 5 (Higher-order)"}

# ── Plot 1: Raw Dataset ──────────────────────
ax1 = axes[0, 0]
ax1.scatter(temperature, sales, color="#3498db", alpha=0.5, edgecolors="white",
            linewidth=0.5, s=40, label="Data Points")
ax1.set_title("Raw Dataset", fontsize=12, fontweight="bold")
ax1.set_xlabel("Temperature (°C)")
ax1.set_ylabel("Ice Cream Sales (units)")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.4)

# ── Plot 2: All 3 Model Fits ─────────────────
ax2 = axes[0, 1]
ax2.scatter(X_train, y_train, color="#3498db", alpha=0.4, s=30, label="Train Data")
ax2.scatter(X_test,  y_test,  color="#e67e22", alpha=0.6, s=40, label="Test Data", marker="D")

for deg, model in models.items():
    y_plot = model.predict(X_plot)
    ax2.plot(X_plot, y_plot, color=colors[deg], linewidth=2.2, label=labels[deg])

ax2.set_title("Model Comparison (All Degrees)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Temperature (°C)")
ax2.set_ylabel("Ice Cream Sales (units)")
ax2.legend(fontsize=8)
ax2.grid(True, linestyle="--", alpha=0.4)

# ── Plot 3: Best Model (Degree 2) Residuals ──
ax3 = axes[1, 0]
best_model = models[2]
y_pred_train = best_model.predict(X_train)
residuals    = y_train - y_pred_train

ax3.scatter(y_pred_train, residuals, color="#2ecc71", alpha=0.6, edgecolors="white",
            linewidth=0.5, s=40)
ax3.axhline(y=0, color="#e74c3c", linewidth=1.5, linestyle="--")
ax3.set_title("Residual Plot — Degree 2 Model", fontsize=12, fontweight="bold")
ax3.set_xlabel("Predicted Sales")
ax3.set_ylabel("Residuals")
ax3.grid(True, linestyle="--", alpha=0.4)

# ── Plot 4: R² Score Comparison Bar Chart ────
ax4 = axes[1, 1]
r2_scores = [r2_score(y_test, m.predict(X_test)) for m in models.values()]
bar_colors = [colors[d] for d in degrees]
bars = ax4.bar([f"Degree {d}" for d in degrees], r2_scores,
               color=bar_colors, edgecolor="white", linewidth=1.2, width=0.5)

for bar, score in zip(bars, r2_scores):
    ax4.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.005,
             f"{score:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax4.set_ylim(0, 1.1)
ax4.set_title("R² Score by Polynomial Degree", fontsize=12, fontweight="bold")
ax4.set_ylabel("R² Score")
ax4.grid(True, axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/polynomial_regression_plots.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Plot saved to polynomial_regression_plots.png")